In [1]:
import pandas as pd 
import numpy as np
from datetime import datetime, date

In [2]:
from aind_data_access_api.document_db import MetadataDbClient

API_GATEWAY_HOST = "api.allenneuraldynamics.org"
DATABASE = 'metadata_index'
COLLECTION = 'data_assets'

docdb_api_client = MetadataDbClient(
   host=API_GATEWAY_HOST,
   database=DATABASE,
   collection=COLLECTION,
)
print(docdb_api_client._base_url)

https://api.allenneuraldynamics.org/v1/metadata_index/data_assets


In [3]:
aggregate = [
  {
    "$match": {
      "data_description.project_name": "V1 Deep Dive"
    }
  },
  {
    "$project": {
      "name": 1, 
      "subject_id": "$data_description.subject_id",
      "genotype": "$subject.subject_details.genotype", 
      "date_of_birth": "$subject.subject_details.date_of_birth", 
      "sex": "$subject.subject_details.sex", 
      "session_time": "$acquisition.acquisition_start_time",
      "project_name": "$data_description.project_name", 
      "modality": "$data_description.modalities.name",
      "column": { "$arrayElemAt": ["$data_description.tags", 0] },
      "volume": { "$arrayElemAt": ["$data_description.tags", 1] }
    }
  },
]
    
records = docdb_api_client.aggregate_docdb_records(
    pipeline = aggregate,
)

In [14]:
len(records)

547

In [13]:
records = [r for r in records if 'Electron microscopy' not in r['modality']]
df = pd.DataFrame(records)

df['session_date'] = df.apply(lambda x: datetime.fromisoformat(x['session_time']).date(), axis=1)
df['session_time'] = df.apply(lambda x: datetime.fromisoformat(x['session_time']).time(), axis=1)
df['date_of_birth'] = df.apply(lambda x: datetime.strptime(x['date_of_birth'], '%Y-%m-%d').date(), axis=1)
df['age'] = df.apply(lambda x: (x['session_date'] - x['date_of_birth']).days, axis=1)

df['column'] = df.apply(lambda x: int(x['column'].split(' ')[-1]), axis=1)
df['volume'] = df.apply(lambda x: int(x['volume'].split(' ')[-1]), axis=1)

df['golden_mouse'] = False
df.loc[df.subject_id=='409828', 'golden_mouse'] = True

order = ['project_name','_id','name','subject_id','golden_mouse','genotype','date_of_birth','sex','modality',
         'session_date','age','session_time','column','volume']
df = df[order]

df.head()

,project_name,_id,name,subject_id,golden_mouse,genotype,date_of_birth,sex,modality,session_date,age,session_time,column,volume
0,V1 Deep Dive,ed4558a4-4b1d-48a0-a257-8b4f455ddede,409828_2018-11-06_14-02-59_nwb_2025-12-15_19-5...,409828,True,Slc17a7-IRES2-Cre/wt;Camk2a-tTA/wt;Ai94(TITL-G...,2018-07-03,Male,"[Planar optical physiology, Behavior videos]",2018-11-06,126,14:02:59.889330,2,1
1,V1 Deep Dive,6eccd622-d0f9-4227-8559-e184aa480268,409828_2018-11-20_10-42-45_nwb_2025-12-15_21-5...,409828,True,Slc17a7-IRES2-Cre/wt;Camk2a-tTA/wt;Ai94(TITL-G...,2018-07-03,Male,"[Planar optical physiology, Behavior videos]",2018-11-20,140,10:42:45.603660,3,1
2,V1 Deep Dive,aa8c08d9-6f3f-4a56-858f-5962ccb62aeb,409828_2018-11-20_12-12-08_nwb_2025-12-15_22-1...,409828,True,Slc17a7-IRES2-Cre/wt;Camk2a-tTA/wt;Ai94(TITL-G...,2018-07-03,Male,"[Planar optical physiology, Behavior videos]",2018-11-20,140,12:12:08.089640,3,2
3,V1 Deep Dive,b064bb9c-5d73-416d-bb38-6862264f0f74,409828_2018-11-21_09-22-23_nwb_2025-12-15_23-2...,409828,True,Slc17a7-IRES2-Cre/wt;Camk2a-tTA/wt;Ai94(TITL-G...,2018-07-03,Male,"[Planar optical physiology, Behavior videos]",2018-11-21,141,09:22:23.259970,4,1
4,V1 Deep Dive,ef64dafa-933a-4d22-b519-2701afce6081,409828_2018-11-21_10-56-07_nwb_2025-12-15_23-4...,409828,True,Slc17a7-IRES2-Cre/wt;Camk2a-tTA/wt;Ai94(TITL-G...,2018-07-03,Male,"[Planar optical physiology, Behavior videos]",2018-11-21,141,10:56:07.266840,4,2


In [22]:
test = df.drop_duplicates(subset=['subject_id','session_date'], inplace=False)
len(test)

60

In [ ]:
# df.to_csv('/data/metadata/V1DD_metadata.csv', index= False)